# Referral Bypass - Initial Data Exploration

This notebook profiles the two source extracts and checks schema, grain, identity coverage, multiplicity, and chronology. It does not publish the final bypass count.

The formal controlled analysis is in `booking_centric_bypass_detection.ipynb`.

## 1. Setup and Load

The notebook identifies the two CSV files by schema and reads IDs as strings.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

# 定位项目目录
CURRENT_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = [CURRENT_DIR, CURRENT_DIR.parent, *list(CURRENT_DIR.parents)[:3]]
PROJECT_ROOT = next((path for path in ROOT_CANDIDATES if (path / "data" / "raw").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot locate data/raw from {CURRENT_DIR}")

SOURCE_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "derived"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    file.name: pd.read_csv(file, dtype="string", low_memory=False)
    for file in sorted(SOURCE_DIR.glob("*.csv"))
}
referral_file = next(name for name, table in tables.items() if "referral_enquiry_id" in table.columns)
booking_file = next(name for name, table in tables.items() if "booking_id" in table.columns)
referrals = tables[referral_file]
bookings = tables[booking_file]

print(f"Referral file: {referral_file} -> {len(referrals):,} rows")
print(f"Booking file: {booking_file} -> {len(bookings):,} rows")

Referral file: referral_submissions.csv -> 5,448 rows
Booking file: booking_extracts.csv -> 5,275 rows


## 2. Grain and Key Quality

The expected grains are one referral enquiry per row and one booking per row.

In [2]:
# 检查表粒度和主键
table_profile = pd.DataFrame([
    {
        "table": "Referral submissions",
        "rows": len(referrals),
        "columns": referrals.shape[1],
        "key": "referral_enquiry_id",
        "non_null_keys": referrals["referral_enquiry_id"].notna().sum(),
        "unique_keys": referrals["referral_enquiry_id"].nunique(dropna=True),
        "duplicate_keys": referrals["referral_enquiry_id"].duplicated().sum(),
    },
    {
        "table": "Bookings",
        "rows": len(bookings),
        "columns": bookings.shape[1],
        "key": "booking_id",
        "non_null_keys": bookings["booking_id"].notna().sum(),
        "unique_keys": bookings["booking_id"].nunique(dropna=True),
        "duplicate_keys": bookings["booking_id"].duplicated().sum(),
    },
])
display(table_profile)

,table,rows,columns,key,non_null_keys,unique_keys,duplicate_keys
0,Referral submissions,5448,29,referral_enquiry_id,5448,5448,0
1,Bookings,5275,23,booking_id,5275,5275,0


## 3. Source and Identity Coverage

`lead_id` is the primary identity key. Exact encrypted mobile/email can be used later as secondary evidence within the same client.

In [3]:
# 查看来源分布
source_distribution = pd.concat([
    bookings["booking_source"].fillna("<NULL>").value_counts(dropna=False).rename_axis("value").reset_index(name="count").assign(field="booking_source"),
    bookings["enquiry_source"].fillna("<NULL>").value_counts(dropna=False).rename_axis("value").reset_index(name="count").assign(field="enquiry_source"),
], ignore_index=True)[["field", "value", "count"]]

# 查看身份字段覆盖率
identity_coverage = pd.DataFrame([
    {"table": "Referrals", "field": field, "non_null_rows": referrals[field].notna().sum(), "unique_values": referrals[field].nunique(dropna=True)}
    for field in ["lead_id", "lead_hash_id", "mobileNumber", "email"]
] + [
    {"table": "Bookings", "field": field, "non_null_rows": bookings[field].notna().sum(), "unique_values": bookings[field].nunique(dropna=True)}
    for field in ["lead_id", "lead_hash_id", "mobileNumber", "email"]
])

display(source_distribution)
display(identity_coverage)

,field,value,count
0,booking_source,REFERRAL,2768
1,booking_source,CHANNEL_PARTNER,1939
2,booking_source,<NULL>,544
3,booking_source,DIRECT,24
4,enquiry_source,REFERRAL,2781
5,enquiry_source,CHANNEL_PARTNER,1919
6,enquiry_source,<NULL>,575


,table,field,non_null_rows,unique_values
0,Referrals,lead_id,5448,1000
1,Referrals,lead_hash_id,5448,1000
2,Referrals,mobileNumber,5448,971
3,Referrals,email,5299,838
4,Bookings,lead_id,4700,701
5,Bookings,lead_hash_id,4700,701
6,Bookings,mobileNumber,4691,688
7,Bookings,email,4575,571


## 4. Lead Multiplicity

Many leads have repeated referrals or bookings. A direct many-to-many join would duplicate records, so the formal notebook selects the latest time-valid prior referral per booking and keeps multiplicity flags.

In [4]:
# 统计每个 lead 的 referral 和 booking 数量
referral_counts = referrals.dropna(subset=["lead_id"]).groupby("lead_id").size().rename("referral_count")
booking_counts = bookings.dropna(subset=["lead_id"]).groupby("lead_id").size().rename("booking_count")
lead_multiplicity = pd.concat([referral_counts, booking_counts], axis=1).fillna(0).astype(int)

multiplicity_summary = pd.DataFrame([
    {"metric": "Unique referral leads", "value": referrals["lead_id"].nunique(dropna=True)},
    {"metric": "Unique booking leads", "value": bookings["lead_id"].nunique(dropna=True)},
    {"metric": "Leads in both tables", "value": int(((lead_multiplicity["referral_count"] > 0) & (lead_multiplicity["booking_count"] > 0)).sum())},
    {"metric": "Leads with multiple referrals", "value": int((lead_multiplicity["referral_count"] > 1).sum())},
    {"metric": "Leads with multiple bookings", "value": int((lead_multiplicity["booking_count"] > 1).sum())},
])
display(multiplicity_summary)
display(lead_multiplicity.sort_values(["referral_count", "booking_count"], ascending=False).head(10))

,metric,value
0,Unique referral leads,1000
1,Unique booking leads,701
2,Leads in both tables,595
3,Leads with multiple referrals,29
4,Leads with multiple bookings,14


,referral_count,booking_count
lead_id,,
893,309,570
890,309,569
884,308,566
887,308,565
896,307,566
899,306,566
902,306,559
883,300,2
886,300,0


## 5. Same-Enquiry Chronology

Chronology is checked before source attribution. A same-enquiry booking that predates referral submission cannot be used as evidence that the referral was bypassed.

In [5]:
# 解析 same-enquiry 时间
def parse_datetime(series):
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(series, errors="coerce")


referral_time = parse_datetime(referrals["referral_submitted_on"])
same_booking_time = parse_datetime(referrals["booking_on_referral_date"])
same_booking_source = referrals["booking_on_referral_source"].fillna("<NULL>").str.strip().str.upper()
same_booking_exists = referrals["booking_on_referral_enquiry_id"].notna()

time_valid_mask = same_booking_exists & same_booking_time.notna() & referral_time.notna() & same_booking_time.ge(referral_time)
reverse_time_mask = same_booking_exists & same_booking_time.notna() & referral_time.notna() & same_booking_time.lt(referral_time)
source_disagreement_mask = same_booking_exists & ~same_booking_source.eq("REFERRAL")

chronology_summary = pd.DataFrame([
    {"metric": "Same-enquiry booking rows", "value": int(same_booking_exists.sum())},
    {"metric": "Time-valid same-enquiry rows", "value": int(time_valid_mask.sum())},
    {"metric": "Reverse-time same-enquiry rows", "value": int(reverse_time_mask.sum())},
    {"metric": "Source-disagreement rows", "value": int(source_disagreement_mask.sum())},
    {"metric": "Source disagreement and reverse time", "value": int((source_disagreement_mask & reverse_time_mask).sum())},
    {"metric": "Time-valid source disagreement", "value": int((source_disagreement_mask & time_valid_mask).sum())},
])
display(chronology_summary)

,metric,value
0,Same-enquiry booking rows,2762
1,Time-valid same-enquiry rows,16
2,Reverse-time same-enquiry rows,2746
3,Source-disagreement rows,28
4,Source disagreement and reverse time,28
5,Time-valid source disagreement,0


## 6. Exploration Handoff

- The extracts have unique primary keys at their stated grains.
- Repeated referrals and bookings make direct many-to-many lead joins unsafe without a selection rule.
- The 28 source-disagreement rows are all reverse-time and belong in data-integrity review.
- The formal notebook applies the test/demo filter, primary booking-first attribution, strict/broad sensitivity, confidence tiers, and secondary identity waterfall.

In [6]:
# 汇总探索结果
# -> 结果保留在 notebook，不重复导出 CSV
initial_exploration_summary = pd.concat([
    table_profile.assign(section="table_profile").rename(columns={"rows": "value", "table": "metric"})[["section", "metric", "value"]],
    multiplicity_summary.assign(section="multiplicity")[["section", "metric", "value"]],
    chronology_summary.assign(section="chronology")[["section", "metric", "value"]],
], ignore_index=True)

display(initial_exploration_summary)
assert int((source_disagreement_mask & time_valid_mask).sum()) == 0
print("Exploration checks passed.")

,section,metric,value
0,table_profile,Referral submissions,5448
1,table_profile,Bookings,5275
2,multiplicity,Unique referral leads,1000
3,multiplicity,Unique booking leads,701
4,multiplicity,Leads in both tables,595
5,multiplicity,Leads with multiple referrals,29
6,multiplicity,Leads with multiple bookings,14
7,chronology,Same-enquiry booking rows,2762
8,chronology,Time-valid same-enquiry rows,16
9,chronology,Reverse-time same-enquiry rows,2746


Exploration checks passed.
